# Transport

> Shared async HTTP transport.

In [ ]:
#| default_exp transport

## Imports

In [ ]:
#| export
from __future__ import annotations
import json
from typing import Any, AsyncIterator, Dict, Optional
import httpx
from .errors import ProtocolError
from .sse import SSEvent, aiter_sse

In [ ]:
#| hide
from fastcore.test import *

## Purpose And Design

`transport.py` provides the minimal async HTTP transport abstraction over `httpx`.

### Why this module exists

The library needs a single place for request execution, content decoding, multipart header handling, and SSE streaming mechanics.

### Architectural fit

- Upstream: operation requests from `oapi`.
- Downstream: decoded JSON/text/binary payloads and parsed SSE events.

### Key design choices

- Keep transport thin: no provider logic here.
- Decode by `content-type` to avoid caller duplication.
- Ensure stream error bodies are readable (`aread()` on status errors) for better diagnostics.

### Reader outcome

After this notebook, you should understand the boundary between generic HTTP mechanics (`transport`) and API semantics (`oapi`/`clients`).

### `AsyncTransport`

Thin async HTTP + SSE transport wrapper.

**Why it exists**
- Centralizes request execution, content decoding, multipart behavior, and SSE streaming mechanics.

**Design Notes**
- Intentionally provider-agnostic.
- Keeps low-level I/O concerns separate from API semantics.

**Connections**
- Used by `OpenAPIClient` and therefore all provider clients.

In [ ]:
#| export
class AsyncTransport:
    "Thin async transport wrapper over httpx."
    def __init__(self, *, timeout: float = 60.0, client: Optional[httpx.AsyncClient] = None,
        base_headers: Optional[Dict[str, str]] = None):
        self._own_client = client is None
        self.client = client or httpx.AsyncClient(timeout=timeout)
        self.base_headers = base_headers or {}

    async def aclose(self):
        if self._own_client: await self.client.aclose()

    def _headers(self, headers: Optional[Dict[str, str]] = None) -> Dict[str, str]:
        return {**self.base_headers, **(headers or {})}

    def _request_headers(self, headers: Optional[Dict[str, str]] = None, *, files: Optional[Any] = None) -> Dict[str, str]:
        "Merge headers and drop explicit content-type for multipart uploads."
        h = self._headers(headers)
        if files is not None:
            # Let httpx compute multipart/form-data with boundary.
            for k in list(h):
                if k.lower() == "content-type":
                    h.pop(k, None)
        return h

    @staticmethod
    def _decode(resp: httpx.Response) -> Any:
        "Decode response body using content type."
        ctype = (resp.headers.get("content-type") or "").lower()
        if "application/json" in ctype or ctype.endswith("+json"):
            return resp.json()
        if ctype.startswith("text/") or "application/x-ndjson" in ctype:
            return resp.text
        return resp.content

    async def request(self, method: str, url: str, *, headers: Optional[Dict[str, str]] = None,
        params: Optional[Dict[str, Any]] = None, json_data: Optional[Any] = None, data: Optional[Any] = None,
        files: Optional[Any] = None, raw: bool = False) -> Any:
        "Execute a request and decode JSON/text/binary response."
        resp = await self.client.request(method, url, headers=self._request_headers(headers, files=files), params=params,
            json=json_data, data=data, files=files)
        resp.raise_for_status()
        return resp if raw else self._decode(resp)

    async def request_json(self, method: str, url: str, *, headers: Optional[Dict[str, str]] = None,
        params: Optional[Dict[str, Any]] = None, json_data: Optional[Any] = None) -> Dict[str, Any]:
        data = await self.request(method, url, headers=headers, params=params, json_data=json_data)
        if not isinstance(data, dict): raise ProtocolError(f"Expected dict JSON response, got {type(data).__name__}")
        return data

    async def stream_sse(self, method: str, url: str, *, headers: Optional[Dict[str, str]] = None,
        params: Optional[Dict[str, Any]] = None, json_data: Optional[Any] = None, data: Optional[Any] = None,
        files: Optional[Any] = None) -> AsyncIterator[SSEvent]:
        async with self.client.stream(method, url, headers=self._request_headers(headers, files=files), params=params, json=json_data,
            data=data, files=files) as resp:
            try:
                resp.raise_for_status()
            except httpx.HTTPStatusError:
                # Ensure body is materialized so callers can read `e.response.text` on stream errors.
                try: await resp.aread()
                except Exception: pass
                raise
            async for event in aiter_sse(resp):
                if not event.data: continue
                yield event

    async def stream_sse_json(self, method: str, url: str, *, headers: Optional[Dict[str, str]] = None,
        params: Optional[Dict[str, Any]] = None, json_data: Optional[Any] = None, data: Optional[Any] = None,
        files: Optional[Any] = None,
        done_token: str = "[DONE]") -> AsyncIterator[Dict[str, Any]]:
        async for event in self.stream_sse(method, url, headers=headers, params=params, json_data=json_data,
            data=data, files=files):
            if event.data == done_token: return
            try: raw = json.loads(event.data)
            except json.JSONDecodeError as e: raise ProtocolError(f"Invalid SSE JSON: {e}") from e
            if isinstance(raw, dict): yield raw
            else: raise ProtocolError(f"Expected SSE JSON object, got {type(raw).__name__}")

## Quick Checks

In [ ]:
import fastllm.transport as m
for nm in ['AsyncTransport']: assert hasattr(m, nm), nm
from fastllm.transport import AsyncTransport
tr = AsyncTransport()
assert hasattr(tr, 'request')
import asyncio
asyncio.run(tr.aclose())